# VEIL — LangChain Reliability Pipeline Demo

End-to-end demo of the `src.pipeline` module:

```
{"article": ...}
   |
   ├─ BART summarizer (with neutral-prompt prefix)
   ├─ FinBERT sentiment classifier
   └─ Loughran–McDonald lexicon signal + bias score
        |
        v
   reliability_assess  →  PipelineOutput (JSON)
```

The demo runs three articles spanning **positive / negative / mixed-bias** so the
`reliability` tier and `reasons` fields can be inspected.

> Run from the repo root so `src` is importable. From `notebooks/` we patch
> `sys.path` in the next cell.

**Environment note (important):** if you see a NumPy compatibility error,
install the dedicated file once in your current env:

```bash
pip install -r requirements-pipeline.txt
```

Then restart the notebook kernel before re-running cells.

In [1]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.pipeline import build_pipeline

pipeline = build_pipeline()
print("pipeline built ✓")

pipeline built ✓


In [2]:
def show(article: str) -> None:
    out = pipeline.invoke({"article": article})
    print(json.dumps(out.model_dump(), indent=2, ensure_ascii=False))
    print("-" * 80)

## Sample 1 — clear positive earnings beat
Expect: `finbert.label == lexicon.label == "positive"`, `reliability == "high"`.

In [3]:
positive_article = (
    "Apple reported record fourth-quarter revenue of $94.9 billion on Thursday, "
    "exceeding analyst expectations and posting strong growth in services and wearables. "
    "Earnings per share rose 13% year over year, and management raised its full-year guidance, "
    "citing robust iPhone demand and continued momentum in emerging markets. "
    "Shares climbed more than 4% in after-hours trading."
)
show(positive_article)

/Users/gala/anaconda3/envs/6731/lib/python3.11/site-packages/transformers/pytorch_utils.py:338: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


{
  "summary": "Apple reported record fourth-quarter revenue of $94.9 billion on Thursday. Earnings per share rose 13% year over year, and management raised its full-year guidance.",
  "finbert": {
    "label": "positive",
    "confidence": 0.9554
  },
  "lexicon": {
    "label": "positive",
    "counts": {
      "positive": 1
    }
  },
  "bias": {
    "uncertainty": 0.0,
    "litigious": 0.0,
    "constraining": 0.0,
    "score": 0.0,
    "flag": false
  },
  "agreement": true,
  "reliability": "high",
  "reasons": []
}
--------------------------------------------------------------------------------


## Sample 2 — clear negative profit warning
Expect: both signals lean negative, `reliability == "high"` (consistent + confident).

In [4]:
negative_article = (
    "Boeing slashed its annual delivery forecast on Wednesday after a fresh round of "
    "production defects and supplier shortages forced the planemaker to halt 737 MAX shipments. "
    "The company warned of a $1.6 billion loss for the quarter, suspended its 2025 financial guidance, "
    "and disclosed an SEC inquiry into its quality controls. Shares plunged 9% on the news, "
    "their worst single-day decline in two years."
)
show(negative_article)

{
  "summary": "Boeing slashed its annual delivery forecast on Wednesday after a fresh round of production defects and supplier shortages. The company warned of a $1.6 billion loss for the quarter. Shares plunged 9% on the news, their worst single-day decline in two years.",
  "finbert": {
    "label": "negative",
    "confidence": 0.9724
  },
  "lexicon": {
    "label": "negative",
    "counts": {
      "negative": 11
    }
  },
  "bias": {
    "uncertainty": 0.0,
    "litigious": 0.0,
    "constraining": 0.0,
    "score": 0.0,
    "flag": false
  },
  "agreement": true,
  "reliability": "high",
  "reasons": []
}
--------------------------------------------------------------------------------


## Sample 3 — speculative / hedged language
Heavy use of `may`, `could`, `uncertain`, `subject to` should push the L&M
**bias score** above the 0.15 threshold and likely demote `reliability` to
`medium` or `low` even if FinBERT is confident.

In [5]:
speculative_article = (
    "Tesla may face additional regulatory scrutiny over its full self-driving claims, "
    "which could result in litigation and possible restrictions on future deployments. "
    "Analysts caution that the outcome remains uncertain and is subject to ongoing investigations "
    "by federal authorities. While management has not commented, restrictions imposed on similar "
    "programs in the past have constrained vehicle deliveries and limited near-term revenue growth."
)
show(speculative_article)

{
  "summary": "Tesla may face additional regulatory scrutiny over its full self-driving claims, which could result in litigation and possible restrictions on future deployments. Analysts caution that the outcome remains uncertain and is subject to ongoing investigations.",
  "finbert": {
    "label": "negative",
    "confidence": 0.9689
  },
  "lexicon": {
    "label": "negative",
    "counts": {
      "uncertainty": 4,
      "litigious": 1,
      "negative": 5,
      "constraining": 4
    }
  },
  "bias": {
    "uncertainty": 0.2857,
    "litigious": 0.0714,
    "constraining": 0.2857,
    "score": 0.2143,
    "flag": true
  },
  "agreement": true,
  "reliability": "medium",
  "reasons": [
    "high_lexicon_bias"
  ]
}
--------------------------------------------------------------------------------
